Concurrency is about dealing with lots of things at once.
Parallelism is about doing lots of things at once.
Not the same, but related.
One is about structure, one is about execution.
Concurrency provides a way to structure a solution to solve a problem that may (but
not necessarily) be parallelizable.
—Rob Pike, co-inventor of the Go language

### Execution units

1. Asyncio / Coroutines (Concurrent, but NOT Parallel)
How it works: Everything runs on one single CPU core, in one single thread. It achieves concurrency through cooperative multitasking. When Task A hits a waiting period (like await download_image()), it voluntarily steps aside so Task B can run.

The Analogy: One chef in the kitchen. He puts the pizza in the oven, sets a timer, and instantly turns around to chop a salad. He is managing two dishes concurrently, but he is only ever doing one action at a exact millisecond in time.

2. Threading (Concurrent, but NOT Parallel)
How it works: You have multiple threads, but Python has a infamous mechanism called the Global Interpreter Lock (GIL). The GIL ensures that only one thread can execute Python code at any given time. The Operating System rapidly pauses and switches between threads, giving the illusion of parallelism, but it's still fundamentally concurrent.
**in normal os there is no GIL and there is preemptive multitasking**

The Analogy: Multiple chefs in one kitchen, but there is only one spatula (the GIL). Whoever holds the spatula gets to cook. The chefs constantly rip the spatula out of each other's hands. It looks like a blur of activity, but only one chef is actually cooking at any exact moment.

3. Multiprocessing (Concurrent AND Parallel)
How it works: Python literally clones your entire program and runs it in entirely separate memory spaces. Because they are separate processes, they bypass the GIL entirely. They can be assigned to different physical cores on your CPU.

The Analogy: You rent three separate kitchens, hire three separate chefs, and give them all their own spatulas. They are truly cooking at the exact same time.

## Threads

In [ ]:
import itertools
from threading import Thread, Event
import time


def spin(msg , done:Event):
    for char in itertools.cycle(r'\|/-'):
        status = f'\r {msg} {char}'
        print(status, end="", flush=True)
        if done.wait(.1):
            break
    blanks = ' ' * len(status)
    print(f'\r{blanks}\r', end='')




def slow(): 
    time.sleep(3)  #remember sleep release the GIL so other threads works
    return 3


def spin(msg: str, done: Event) -> None:
    """
    The 'Worker' thread function.
    Runs a continuous loop until the 'done' event is flagged.
    """
    # itertools.cycle creates an infinite loop of the characters \|/-
    for char in itertools.cycle(r'\|/-'):
        status = f'\r {msg} {char}'
        # \r moves the cursor back to the start of the line so it overwrites itself
        print(status, end="", flush=True)
        
        # This is the "brain" of the loop:
        # .wait(.1) pauses for 0.1s OR until 'done' is set.
        # If 'done' is set, it returns True and we break the loop.
        if done.wait(.1):
            break
            
    # Clean up the line by overwriting it with blanks once finished
    blanks = ' ' * len(status)
    print(f'\r{blanks}\r', end='')

def slow() -> int:
    """
    Simulates a heavy task.
    """
    #  time.sleep releases the GIL (Global Interpreter Lock).
    # This allows the 'spin' thread to take over the CPU while this thread rests.
    time.sleep(3)  
    return 3

def supervisor() -> int:
    """
    The 'Manager' function that coordinates the threads.
    """
    done = Event() # The communication bridge (False by default)
    
    # Create the thread, passing the 'done' event so 'spin' can watch it
    spinner = Thread(target=spin, args=('thinking!', done))
    print(f'spinner object: {spinner}')
    
    spinner.start() # Starts the spin() function in the background
    
    result = slow() # The main thread blocks here for 3 seconds
    
    done.set()      # 'slow' is done! Flip the flag to True to stop the spinner
    spinner.join()  # Wait for the spinner to finish its last print/cleanup
    
    return result

def main() -> None:
    result = supervisor()
    print(f'Answer: {result}')

if __name__ == '__main__':
    main()


spinner object: <Thread(Thread-105 (spin), initial)>
Answer: 3    


In [ ]:
import itertools
import time
from threading import Thread, Event

def server_entertainment(msg: str, order_ready: Event):
    """The Spinner Thread: Keeps the customer happy."""
    for char in itertools.cycle(['Checking status...', 'Pouring water...', 'Bringing bread...']):
        # \r keeps the status on the same line at the table
        print(f'\r{char}', end="", flush=True)
        
        # Check if the Chef has rung the bell (order_ready)
        if order_ready.wait(2): # Wait 1 second between actions
            break
            
    print(f"\n[Server]: 'Your meal is here! Enjoy!'")

def chef_cooking():
    """The 'Slow' Function: The main task that takes time."""
    print("[Chef]: Starting the 3-minute risotto...")
    time.sleep(10) # The heavy lifting
    return "Delicious Risotto"

def restaurant_supervisor():
    """The Manager: Coordinates the Chef and the Server."""
    bell = Event() # The communication signal between kitchen and floor
    
    # Send the server out to the table
    waiter_thread = Thread(target=server_entertainment, args=('Service', bell))
    waiter_thread.start()
    
    # Chef starts cooking (This blocks the supervisor's main attention)
    meal = chef_cooking()
    
    # Chef rings the bell!
    bell.set()  #command to stop
    
    # Wait for the waiter to stop their current action and come back
    waiter_thread.join() #confirmation of stooping
    
    return meal

# Execution
if __name__ == '__main__':
    final_dish = restaurant_supervisor()
    print(f"Table received: {final_dish}")

Checking status...[Chef]: Starting the 3-minute risotto...
Pouring water.....
[Server]: 'Your meal is here! Enjoy!'
Table received: Delicious Risotto


## Process

In [15]:
import itertools
import time
from multiprocessing import Process, Event

def spin(msg: str, done: Event) -> None:
    # In Multiprocessing, this function is running in its own memory space
    for char in itertools.cycle(r'\|/-'):
        status = f'\r {msg} {char}'
        print(status, end="", flush=True)
        if done.wait(.1):
            break
    
    blanks = ' ' * len(status)
    print(f'\r{blanks}\r', end='')

def slow() -> int:
    # This runs in the Main Process. 
    # It no longer cares about the GIL because the spinner is in another process.
    time.sleep(3)  
    return 3

def supervisor() -> int:
    # Multiprocessing Events are designed to work across different memory spaces
    done = Event() 
    
    # We use Process instead of Thread
    spinner = Process(target=spin, args=('thinking!', done))
    
    spinner.start() 
    result = slow() 
    
    done.set()      
    spinner.join()  
    
    return result

def main() -> None:
    result = supervisor()
    print(f'Answer: {result}')

if __name__ == '__main__':
    main()

Answer: 3


because multiprocessing have boundries between the process some might not show like here on jupyter note book

The multiprocessing.shared_memory module (introduced in Python 3.8) is the "pro-tier" way to handle shared data. Unlike Value or Array, which are wrapped in high-level Python objects, shared_memory creates a raw block of bytes in your operating system's RAM that both processes can "see" at the same address.

It is incredibly fast because there is no pickling (serializing data) and no "manager" process acting as a middleman.

## Coroutines

In [ ]:
# THIS WOULD ONLY RUN ON .py
import asyncio

# The main entry point - the only 'regular' function
def main() -> None:
    # asyncio.run starts the event loop and drives the supervisor coroutine
    # It will block here until supervisor() returns its value
    result = asyncio.run(supervisor())
    print(f'Answer: {result}')

# Native coroutines are defined with 'async def'
async def supervisor() -> int:
    # 1. Schedule 'spin' to run eventually.
    # This immediately returns a Task object (the juggler starts).
    spinner = asyncio.create_task(spin('thinking!'))
    print(f'spinner object: {spinner}')

    # 2. The 'await' keyword pauses supervisor() and gives control back
    # to the event loop, allowing 'spin' to run while we wait for 'slow'.
    result = await slow()

    # 3. Once 'slow' returns, we manually stop the spinner.
    # This raises a CancelledError inside the spin coroutine.
    spinner.cancel()

    return result

# You would also have your spin and slow coroutines defined like this:
# async def spin(msg): ...
# async def slow(): ...

if __name__ == '__main__':
    main()

#this in not concureent in a way to save time, after we gonna learn how to create task and use em

In [ ]:
import asyncio
import itertools

async def spin(msg: str) -> None:
    for char in itertools.cycle(r'\|/-'):
        status = f'\r{char} {msg}'
        print(status, flush=True, end='')
        try:
            await asyncio.sleep(.1)
        except asyncio.CancelledError:
            break
    blanks = ' ' * len(status)
    print(f'\r{blanks}\r', end='')

async def slow() -> int:
    await asyncio.sleep(3)
    return 42

async def supervisor() -> int:
    # 1. Schedule the spinner to run in the background immediately
    spinner_task = asyncio.create_task(spin('thinking!'))
    
    # 2. Block this specific coroutine until slow() finishes
    # Meanwhile, the Event Loop will run the spinner_task
    result = await slow()
    
    # 3. slow() is done! Throw the CancelledError into the spinner
    spinner_task.cancel()
    
    return result

def main() -> None:
    # asyncio.run() creates the Event Loop, runs the supervisor, and closes the loop cleanly
    result = asyncio.run(supervisor())
    print(f'Answer: {result}')

if __name__ == '__main__':
    main()